In [1]:
import os
import requests
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")
headers = {"Authorization": f"Bearer {os.environ['CFBD_KEY']}"}
BASE = "https://api.collegefootballdata.com"
YEAR = 2026
MIN_PLAYS = 100   # starters below this get flagged for review

In [2]:
CONFERENCES = {
    "SEC": ["SEC"],
    "Big Ten": ["B1G", "Big Ten"],
    "Big 12": ["B12", "Big 12"],
    "ACC": ["ACC"],
}

def get_conference(endpoint, params, names):
    """Call an endpoint for one conference, trying each name until data comes back."""
    for name in names:
        r = requests.get(f"{BASE}{endpoint}",
                         params={**params, "conference": name},
                         headers=headers)
        r.raise_for_status()
        data = r.json()
        if data:
            return data
    return []

In [3]:
frames = []
for label, names in CONFERENCES.items():
    data = get_conference("/ppa/players/season",
                          {"year": YEAR, "position": "QB", "excludeGarbageTime": "true"},
                          names)
    df = pd.json_normalize(data)
    df["conf_label"] = label
    frames.append(df)
    print(f"{label}: {len(df)} QBs")   # a 0 means the conference name didn't match

qbs = pd.concat(frames, ignore_index=True).drop_duplicates(subset="id")
qbs.to_csv("../data/p4_qb_ppa_raw.csv", index=False)

SEC: 25 QBs
Big Ten: 33 QBs
Big 12: 28 QBs
ACC: 29 QBs


In [4]:
stat_frames = []
for label, names in CONFERENCES.items():
    data = get_conference("/stats/player/season", {"year": YEAR}, names)
    stat_frames.append(pd.json_normalize(data))

stats = pd.concat(stat_frames, ignore_index=True)
stats.to_csv("../data/p4_player_stats_raw.csv", index=False)

In [5]:
rec_frames = []
for label, names in CONFERENCES.items():
    data = get_conference("/records", {"year": YEAR}, names)
    rec_frames.append(pd.json_normalize(data))

records = pd.concat(rec_frames, ignore_index=True)
records.to_csv("../data/p4_records_raw.csv", index=False)
records.columns   # confirm the wins/losses/games column names before step 5

Index(['year', 'teamId', 'team', 'classification', 'conference', 'division',
       'expectedWins', 'total.games', 'total.wins', 'total.losses',
       'total.ties', 'conferenceGames.games', 'conferenceGames.wins',
       'conferenceGames.losses', 'conferenceGames.ties', 'homeGames.games',
       'homeGames.wins', 'homeGames.losses', 'homeGames.ties',
       'awayGames.games', 'awayGames.wins', 'awayGames.losses',
       'awayGames.ties', 'neutralSiteGames.games', 'neutralSiteGames.wins',
       'neutralSiteGames.losses', 'neutralSiteGames.ties',
       'regularSeason.games', 'regularSeason.wins', 'regularSeason.losses',
       'regularSeason.ties', 'postseason.games', 'postseason.wins',
       'postseason.losses', 'postseason.ties'],
      dtype='str')

In [6]:
stats["stat"] = pd.to_numeric(stats["stat"], errors="coerce")
attempts = stats[
    ((stats["category"] == "passing") & (stats["statType"] == "ATT")) |
    ((stats["category"] == "rushing") & (stats["statType"] == "CAR"))
]
plays = (attempts.pivot_table(index="playerId", columns="category",
                              values="stat", aggfunc="sum")
                 .fillna(0)
                 .reset_index())
plays["total_plays"] = plays["passing"] + plays["rushing"]

qbs["id"] = qbs["id"].astype(str)
plays["playerId"] = plays["playerId"].astype(str)

merged = qbs.merge(plays, left_on="id", right_on="playerId", how="left")
starters = (merged.sort_values("passing", ascending=False)
                  .drop_duplicates(subset="team"))
print(starters["conf_label"].value_counts())   # expect roughly 16-18 per conference

conf_label
Big Ten    18
ACC        17
Big 12     16
SEC        16
Name: count, dtype: int64


In [7]:
rec = records[["team", "total.games", "total.wins", "total.losses"]].copy()
rec["win_pct"] = rec["total.wins"] / rec["total.games"]

df = starters.merge(rec, on="team", how="left")
print("Teams missing a record:", df.loc[df["win_pct"].isna(), "team"].tolist())

Teams missing a record: []


In [8]:
df = df.dropna(subset=["averagePPA.all", "win_pct"]).copy()

slope, intercept = np.polyfit(df["averagePPA.all"], df["win_pct"], 1)
df["expected_win_pct"] = slope * df["averagePPA.all"] + intercept
df["residual"] = df["win_pct"] - df["expected_win_pct"]
df["low_sample"] = df["total_plays"] < MIN_PLAYS

r = df["averagePPA.all"].corr(df["win_pct"])
print(f"Correlation (r) between PPA and win %: {r:.2f}")
print(f"Trend line: win% = {slope:.2f} x PPA + {intercept:.2f}")

Correlation (r) between PPA and win %: 0.53
Trend line: win% = 0.69 x PPA + 0.45


In [9]:
cols = ["name", "team", "conf_label", "averagePPA.all", "win_pct",
        "expected_win_pct", "residual", "passing", "rushing", "total_plays", "low_sample"]

print("Most 'carrying' (QB better than the record):")
display(df.sort_values("residual")[cols].head(8))

print("Most 'carried' (record better than the QB):")
display(df.sort_values("residual", ascending=False)[cols].head(8))

Most 'carrying' (QB better than the record):


,name,team,conf_label,averagePPA.all,win_pct,expected_win_pct,residual,passing,rushing,total_plays,low_sample
66,Dylan Lonergan,Rutgers,Big Ten,0.325,0.200000,0.672362,-0.472362,92.0,16.0,108.0,False
5,Davis Warren,Stanford,ACC,0.240,0.333333,0.613960,-0.280627,195.0,21.0,216.0,False
2,Ryan Browne,Purdue,Big Ten,0.372,0.428571,0.704655,-0.276084,221.0,40.0,261.0,False
33,Alberto Mendoza,Georgia Tech,ACC,0.287,0.400000,0.646253,-0.246253,153.0,21.0,174.0,False
32,Lincoln Kienholz,Louisville,ACC,0.418,0.500000,0.736261,-0.236261,154.0,61.0,215.0,False
41,Isaiah Marshall,Kansas,Big 12,0.245,0.400000,0.617395,-0.217395,141.0,60.0,201.0,False
11,Katin Houser,Illinois,Big Ten,0.377,0.500000,0.708091,-0.208091,184.0,30.0,214.0,False
12,KJ Jackson,Arkansas,SEC,0.126,0.333333,0.535632,-0.202299,184.0,48.0,232.0,False


Most 'carried' (record better than the QB):


,name,team,conf_label,averagePPA.all,win_pct,expected_win_pct,residual,passing,rushing,total_plays,low_sample
57,Nico Iamaleava,UCLA,Big Ten,0.042,1.000000,0.477917,0.522083,116.0,42.0,158.0,False
40,Will Hammond,Texas Tech,Big 12,0.247,1.000000,0.618770,0.381230,142.0,20.0,162.0,False
63,Faizon Brandon,Tennessee,SEC,0.150,0.833333,0.552122,0.281211,101.0,53.0,154.0,False
49,Devon Dampier,Utah,Big 12,0.403,1.000000,0.725955,0.274045,131.0,51.0,182.0,False
14,Kenny Minchey,Kentucky,SEC,0.193,0.833333,0.581667,0.251666,178.0,32.0,210.0,False
54,Hank Brown,Iowa,Big Ten,0.197,0.833333,0.584415,0.248918,124.0,17.0,141.0,False
58,Bear Bachmeier,BYU,Big 12,0.448,1.000000,0.756874,0.243126,116.0,36.0,152.0,False
38,Arch Manning,Texas,SEC,0.170,0.800000,0.565864,0.234136,147.0,38.0,185.0,False


In [10]:
out = (df[cols]
       .rename(columns={"averagePPA.all": "ppa_per_play",
                        "passing": "pass_att", "rushing": "rush_att"})
       .sort_values("residual"))
out.to_csv("../data/p4_qb_ppa_vs_winpct.csv", index=False)
out.head()

,name,team,conf_label,ppa_per_play,win_pct,expected_win_pct,residual,pass_att,rush_att,total_plays,low_sample
66,Dylan Lonergan,Rutgers,Big Ten,0.325,0.200000,0.672362,-0.472362,92.0,16.0,108.0,False
5,Davis Warren,Stanford,ACC,0.240,0.333333,0.613960,-0.280627,195.0,21.0,216.0,False
2,Ryan Browne,Purdue,Big Ten,0.372,0.428571,0.704655,-0.276084,221.0,40.0,261.0,False
33,Alberto Mendoza,Georgia Tech,ACC,0.287,0.400000,0.646253,-0.246253,153.0,21.0,174.0,False
32,Lincoln Kienholz,Louisville,ACC,0.418,0.500000,0.736261,-0.236261,154.0,61.0,215.0,False


In [11]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df["averagePPA.all"], df["win_pct"], alpha=0.7)
xs = np.linspace(df["averagePPA.all"].min(), df["averagePPA.all"].max(), 50)
ax.plot(xs, slope * xs + intercept, linestyle="--", color="gray")
for _, row in pd.concat([df.nsmallest(4, "residual"), df.nlargest(4, "residual")]).iterrows():
    ax.annotate(row["name"], (row["averagePPA.all"], row["win_pct"]), fontsize=8)
ax.set_xlabel("PPA per play")
ax.set_ylabel("Team win %")
ax.set_title("P4 starting QBs: PPA vs. team win %")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'